In [0]:
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, TargetEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from mlflow.models import infer_signature

In [0]:
gold_df = spark.table("project.gold.delivery_delay_enhanced")

In [0]:
{col: gold_df.filter(F.col(col).isNull()).count() for col in gold_df.columns}

{'order_id': 0,
 'is_late': 0,
 'customer_state': 0,
 'purchase_day_of_week': 0,
 'is_weekend_purchase': 0,
 'estimated_delivery_month': 0,
 'approval_delay_days': 0,
 'estimated_delivery_buffer_days': 0,
 'expected_delivery_duration_days': 0,
 'seller_shipment_buffer_days': 0,
 'freight_ratio': 0,
 'total_order_items': 0,
 'seller_popularity_30d': 0,
 'product_weight_g': 16,
 'product_volume_cm3': 16,
 'product_density': 16,
 'is_bulky_item': 0,
 'product_category_name_english': 0,
 'is_boleto': 0,
 'installments': 0,
 'is_multi_payment': 0}

In [0]:
df = gold_df.dropna()

{col: df.filter(F.col(col).isNull()).count() for col in df.columns}

{'order_id': 0,
 'is_late': 0,
 'customer_state': 0,
 'purchase_day_of_week': 0,
 'is_weekend_purchase': 0,
 'estimated_delivery_month': 0,
 'approval_delay_days': 0,
 'estimated_delivery_buffer_days': 0,
 'expected_delivery_duration_days': 0,
 'seller_shipment_buffer_days': 0,
 'freight_ratio': 0,
 'total_order_items': 0,
 'seller_popularity_30d': 0,
 'product_weight_g': 0,
 'product_volume_cm3': 0,
 'product_density': 0,
 'is_bulky_item': 0,
 'product_category_name_english': 0,
 'is_boleto': 0,
 'installments': 0,
 'is_multi_payment': 0}

In [0]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

In [0]:
gold_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- is_late: integer (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- purchase_day_of_week: integer (nullable = true)
 |-- is_weekend_purchase: integer (nullable = true)
 |-- estimated_delivery_month: integer (nullable = true)
 |-- approval_delay_days: integer (nullable = true)
 |-- estimated_delivery_buffer_days: integer (nullable = true)
 |-- expected_delivery_duration_days: integer (nullable = true)
 |-- seller_shipment_buffer_days: integer (nullable = true)
 |-- freight_ratio: double (nullable = true)
 |-- total_order_items: long (nullable = true)
 |-- seller_popularity_30d: long (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_volume_cm3: integer (nullable = true)
 |-- product_density: double (nullable = true)
 |-- is_bulky_item: integer (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- is_boleto: integer (nullable = true)
 |-- installments: integer (n

In [0]:
categorical_cols = ["customer_state", "product_category_name_english"]

In [0]:
numerical_cols = [
    "purchase_day_of_week", 
    "is_weekend_purchase", 
    "estimated_delivery_month",
    "approval_delay_days", 
    "estimated_delivery_buffer_days", 
    "expected_delivery_duration_days",
    "seller_shipment_buffer_days", 
    "freight_ratio", 
    "total_order_items",
    "seller_popularity_30d", 
    "product_weight_g", 
    "product_volume_cm3",
    "product_density", 
    "is_bulky_item", 
    "is_boleto", 
    "installments", 
    "is_multi_payment"
]

In [0]:
stages = []

## Indexing and Encoding categorical features

In [0]:
for col in categorical_cols:
    indexer = StringIndexer(
        inputCol=col, 
        outputCol=f"{col}_indexed",
        handleInvalid="keep"
    )
    stages.append(indexer)
 
# Target encode the indexed features
for col in categorical_cols:
    target_encoder = TargetEncoder(
        inputCol=f"{col}_indexed",
        outputCol=f"{col}_encoded",
        labelCol="is_late"
    )
    stages.append(target_encoder)

## Assembling all features into a single vector

In [0]:
# Now we have: encoded categorical features (numeric) + numerical features
assembler_inputs = [f"{col}_encoded" for col in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages.append(assembler)

## We use MLflow to track parameters like number of trees and resulting metrics

In [0]:
import os

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/project/gold/tmp"

In [0]:
# Step 3: Add the classifier
rf = RandomForestClassifier(
    labelCol="is_late", 
    featuresCol="features", 
    numTrees=20, 
    maxDepth=10,
    maxBins=128,
    seed=42
)
stages.append(rf)

In [0]:
with mlflow.start_run(run_name="RandomForest_TargetEncoding_Run"):
    
    # Build Pipeline
    pipeline = Pipeline(stages=stages)
 
    # Fit Model
    print("Training model...")
    model = pipeline.fit(train_df)
 
    # Make Predictions
    predictions = model.transform(test_df)
 
    # --- EVALUATION ---
    evaluator_roc = BinaryClassificationEvaluator(
        labelCol="is_late", 
        rawPredictionCol="rawPrediction"
    )
    auc = evaluator_roc.evaluate(predictions)
 
    evaluator_multi = MulticlassClassificationEvaluator(
        labelCol="is_late", 
        predictionCol="prediction"
    )
    f1 = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "f1"})
    recall = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "weightedRecall"})
    precision = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "weightedPrecision"})
 
    # Log parameters and metrics to MLflow
    mlflow.log_param("numTrees", 20)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("encoding_method", "TargetEncoding")
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)
 
    # Log the Model
    mlflow.spark.log_model(model, "model",dfs_tmpdir='/Volumes/project/gold/tmp')
 
    print(f"Training Complete.")
    print(f"Metrics -> AUC: {auc:.4f}, F1-Score: {f1:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}")

Training model...


2026/03/13 14:05:46 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/13 14:05:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-fbaac539-9a7a-41bd-99f5-8c/tmpyubhxfj1/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/13 14:05:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training Complete.
Metrics -> AUC: 0.7297, F1-Score: 0.8834, Recall: 0.9199, Precision: 0.9146


In [0]:
print("\nSample Predictions for Delayed Orders:")
display(predictions.select("order_id", "is_late", "probability", "prediction")
                  .filter((F.col("is_late") == 1) & (F.col("prediction") == 1))
                  .limit(5))


Sample Predictions for Delayed Orders:


order_id,is_late,probability,prediction
0e784a85d6f7aa0ffe9a41d81f5a4ecf,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3445226809495305"",""0.6554773190504695""]}",1.0
17635bd7488306e5da70e0c945e5fe83,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4428774645513194"",""0.5571225354486807""]}",1.0
17635bd7488306e5da70e0c945e5fe83,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4428774645513194"",""0.5571225354486807""]}",1.0
1a7b59b9cddabc898f2cc56389b5fbc1,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2930454128133131"",""0.7069545871866869""]}",1.0
1a9543c90f188e2e4fb14327ad4a9c9b,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4322853158984133"",""0.5677146841015868""]}",1.0


In [0]:
predictions.write.format("delta").mode("overwrite").saveAsTable("project.gold.rf_predictions")

In [0]:
model_name = "project.gold.olist_delivery_delay_predictor"

with mlflow.start_run(run_name="RF_Target_Encoding_Final") as run:
    
    pipeline = Pipeline(stages=stages)

    # Fit Model
    print("Training model (MaxBins=128 optimized)...")
    model = pipeline.fit(train_df)

    # Make Predictions
    predictions = model.transform(test_df)

    # --- EVALUATION ---
    evaluator_roc = BinaryClassificationEvaluator(labelCol="is_late", rawPredictionCol="rawPrediction")
    auc = evaluator_roc.evaluate(predictions)

    evaluator_multi = MulticlassClassificationEvaluator(labelCol="is_late", predictionCol="prediction")
    f1 = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "f1"})
    recall = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "weightedRecall"})

    # Log to MLflow
    mlflow.log_param("numTrees", 20)
    mlflow.log_param("maxBins", 128)
    mlflow.log_param("maxDepth", 12)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("recall", recall)

    # --- 5. CREATE SIGNATURE & INPUT EXAMPLE ---
    # Unity Catalog requires a model signature (input/output specs)
    input_sample = train_df.select(numerical_cols + categorical_cols).limit(5).toPandas()
    output_sample = predictions.select("prediction").limit(5).toPandas()
    signature = infer_signature(input_sample, output_sample)

    # --- 6. LOG AND REGISTER MODEL ---
    # We use artifact_path="model" and provide the inferred signature
    mlflow.spark.log_model(
        spark_model=model, 
        artifact_path="model",
        dfs_tmpdir="/Volumes/project/gold/tmp",
        signature=signature,
        input_example=input_sample
    )

    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"
    print(f"Registering model: {model_name}")
    mlflow.register_model(model_uri, model_name)

    print(f"Training and Registration Complete. AUC: {auc:.4f}, F1: {f1:.4f}")


Training model (MaxBins=128 optimized)...


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/13 14:19:09 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==

Registering model: project.gold.olist_delivery_delay_predictor


Successfully registered model 'project.gold.olist_delivery_delay_predictor'.
Created version '1' of model 'project.gold.olist_delivery_delay_predictor'.


Training and Registration Complete. AUC: 0.7297, F1: 0.8834


In [0]:
# --- 5. RESULTS PREVIEW ---
print("\nSample Predictions for Delayed Orders:")
display(predictions.select("order_id", "is_late", "probability", "prediction")
                  .filter((F.col("is_late") == 1) & (F.col("prediction") == 1))
                  .limit(5))


Sample Predictions for Delayed Orders:


order_id,is_late,probability,prediction
0e784a85d6f7aa0ffe9a41d81f5a4ecf,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.3445226809495305"",""0.6554773190504695""]}",1.0
17635bd7488306e5da70e0c945e5fe83,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4428774645513194"",""0.5571225354486807""]}",1.0
17635bd7488306e5da70e0c945e5fe83,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4428774645513194"",""0.5571225354486807""]}",1.0
1a7b59b9cddabc898f2cc56389b5fbc1,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2930454128133131"",""0.7069545871866869""]}",1.0
1a9543c90f188e2e4fb14327ad4a9c9b,1,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4322853158984133"",""0.5677146841015868""]}",1.0


In [0]:
model_name = "project.gold.olist_delivery_delay_predictor"

with mlflow.start_run(run_name="RF_Target_Encoding_Unity_Catalog_Final") as run:
    
    # --- PIPELINE STAGES DEFINITION ---
    # These are defined inside the run to ensure all logic is localized to this specific experiment
    stages = []

    # Stage 1: StringIndexer (handle high cardinality)
    for col in categorical_cols:
        indexer = StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
        stages.append(indexer)

    # Stage 2: TargetEncoder (captures risk probability per category)
    target_encoder = TargetEncoder(
        inputCols=[f"{col}_indexed" for col in categorical_cols],
        outputCols=[f"{col}_target_encoded" for col in categorical_cols],
        labelCol="is_late",
        handleInvalid="keep"
    )
    stages.append(target_encoder)

    # Stage 3: VectorAssembler (merging features)
    encoded_cols = [f"{col}_target_encoded" for col in categorical_cols]
    assembler_inputs = encoded_cols + numerical_cols
    assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="skip")
    stages.append(assembler)

    # Stage 4: RandomForest Classifier (maxBins=128 for 75+ categories)
    rf = RandomForestClassifier(
        labelCol="is_late", 
        featuresCol="features", 
        numTrees=30, 
        maxBins=128, 
        maxDepth=12,
        seed=42
    )
    stages.append(rf)

    # Create the end-to-end Pipeline
    pipeline = Pipeline(stages=stages)

    # Fit Model - This executes all stages on the train_df
    print("Training end-to-end pipeline...")
    model = pipeline.fit(train_df)

    # Make Predictions on test_df
    print("Applying model to test data...")
    predictions = model.transform(test_df)

    # --- 3. EVALUATION ---
    evaluator_roc = BinaryClassificationEvaluator(labelCol="is_late", rawPredictionCol="rawPrediction")
    auc = evaluator_roc.evaluate(predictions)

    evaluator_multi = MulticlassClassificationEvaluator(labelCol="is_late", predictionCol="prediction")
    f1 = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "f1"})
    recall = evaluator_multi.evaluate(predictions, {evaluator_multi.metricName: "weightedRecall"})

    # Log metrics to MLflow
    mlflow.log_param("numTrees", 30)
    mlflow.log_param("maxBins", 128)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("recall", recall)

    # --- 4. SIGNATURE AND INPUT EXAMPLE (Required for Unity Catalog) ---
    # We sample the raw input columns (String and Numeric) as they appear in the source data
    input_sample = train_df.select(numerical_cols + categorical_cols).limit(5).toPandas()
    output_sample = predictions.select("prediction").limit(5).toPandas()
    signature = infer_signature(input_sample, output_sample)

    # --- 5. LOG AND REGISTER MODEL ---
    # Log the complete PipelineModel with signature and input example
    mlflow.spark.log_model(
        spark_model=model, 
        artifact_path="model",
        dfs_tmpdir="/Volumes/project/gold/tmp",
        signature=signature,
        input_example=input_sample
    )

    # Explicitly register the model version in Unity Catalog
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"
    print(f"Registering model version for: {model_name}")
    mlflow.register_model(model_uri, model_name)

    print(f"Training and Registration Complete. AUC: {auc:.4f}, F1: {f1:.4f}")


Training end-to-end pipeline...
Applying model to test data...


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/13 14:28:32 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==

Registering model version for: project.gold.olist_delivery_delay_predictor


Registered model 'project.gold.olist_delivery_delay_predictor' already exists. Creating a new version of this model...
Created version '2' of model 'project.gold.olist_delivery_delay_predictor'.


Training and Registration Complete. AUC: 0.7510, F1: 0.8858
